# forward 1d

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elma16/beamax/blob/main/examples/forward/forward-1d.ipynb)

> Select **Runtime → Change runtime type → GPU or TPU** in Colab to demonstrate the hardware-acceleration story.


In [ ]:
# Install beamax and its dependencies for Google Colab.
# Safe to skip when running locally from a checkout.
%%capture
%pip install --quiet "beamax[kwave] @ git+https://github.com/elma16/beamax.git"


In [ ]:
%load_ext autoreload
%autoreload 2

import jax.numpy as jnp
import jax as jax
from time import time
from pathlib import Path
# import scipy.io as sio

from beamax import geometry, plotter, transforms, utils
from beamax.decomposition import DyadicDecomposition
from beamax.transforms import MSWPT
from beamax.gb import gb_solvers, SolverConfig
from beamax.solvers import MSGBSolver, KWaveSolver, ShardingStrategy
from beamax.solvers.msgb_solvers import forward_solver_utils
from beamax.plotter import use_beamax_style

from matplotlib.colors import Normalize, TwoSlopeNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib import patheffects as pe
import numpy as np

from kwave.options.simulation_execution_options import SimulationExecutionOptions
from kwave.options.simulation_options import SimulationOptions

jax.config.update("jax_enable_x64", True)

pltgb = plotter.PlotHelper()

ROOT_DIR = utils.detect_root()
PLOT_DIR = Path(ROOT_DIR / "plots")
DATA_DIR = Path(ROOT_DIR / "data")
PLOT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

import matplotlib.pyplot as plt

use_beamax_style()

jax.config.update("jax_enable_x64", False)
jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
jax.config.update(
    "jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir"
)

# Domain setup

In [ ]:
d = 1
N = (256,) * d
dx = (1 / N[0],) * d
periodic = (True,) * d
box_aspect_ratio = (1,)
num_levels = 2
num_boxes_levels = tuple([2 ** (i + 2) for i in range(num_levels)])


def c(x):
    return 1 + 0 * x[..., 0]


windowing = "rectangular_mirror"
input_type = "spatial"
output_type = "spatial"
redundancy = 2

cfl = 0.3

domain = geometry.Domain(N=N, dx=dx, c=c, cfl=cfl, periodic=periodic)
XY, KXY = domain.generate_meshgrid()

KXY = jnp.stack(KXY, axis=-1)

ts = domain.generate_time_domain()
Nt = len(ts)
dt = ts[1] - ts[0]

t1 = time()
dyadic_decomp = DyadicDecomposition(num_levels, N, num_boxes_levels, box_aspect_ratio)
wpt = MSWPT(dyadic_decomp, redundancy, windowing)
wptNone = MSWPT(dyadic_decomp, redundancy, "none")
t2 = time()
print("Time to create params", t2 - t1)

# pltgb.plot_centers(dyadic_decomp.centres_ndim)

binary_mask = jnp.ones(N)
sensors = geometry.Sensor(domain, binary_mask=binary_mask)

In [ ]:
# #######################################
# ### wave packets ######################
# #######################################
boxhf = 3
boxlf = 3
khf = jnp.array([3])
klf = jnp.array([9])
kerft_hf = transforms.compute_frames(dyadic_decomp, boxhf, khf, KXY, redundancy, "none")
kerft_lf = transforms.compute_frames(dyadic_decomp, boxlf, klf, KXY, redundancy, "none")
p0 = utils.unitary_ifft(kerft_hf) + utils.unitary_ifft(kerft_lf)
p0 = p0 / jnp.max(jnp.abs(p0))


#######################################
### POINT SOURCE ######################
#######################################

# p0 = jnp.zeros(N).squeeze()
# p0 = p0.at[N[0] // 4].set(1)


p0 = p0.real
dpdt = jnp.zeros_like(p0).real
coeffs = wpt.forward(p0, input_type="spatial")
coeffs_array = wpt.convert_to_array(coeffs)


tau = 0.01
beam_budget = None
K = utils.choose_K_by_tau(
    coeffs,
    p0,
    wptNone,
    dyadic_decomp,
    wpt,
    tau=tau,
    Kmin=512,
    Kmax=None,
    num_steps=10,
    beam_budget=beam_budget,
)
print(f"[policy] chosen K={K} for τ={tau:.2%}")

indices, coeff_vals = utils.select_levelaware_topK_indices(
    coeffs, dyadic_decomp, wpt, K
)
print("Selected coefficients:", int(indices.size))
threshold = int(indices.size)

data_recon = utils.reconstruct_from_selection(
    coeffs, indices, coeff_vals, wptNone, output_type="spatial"
)
total_coeffs = 2 * N[0]
strategy = "top_n"
threshold = int(total_coeffs * 0.5)
indices, coeff_vals = forward_solver_utils.threshold_coefficients(
    coeffs, threshold, strategy
)
print("Number of coefficients:", len(indices))


thresholded_coeffs = jnp.zeros_like(coeffs)
thresholded_coeffs = thresholded_coeffs.at[indices].set(coeff_vals)

data_recon = wptNone.inverse(thresholded_coeffs, output_type="spatial")

plt.figure()
plt.subplot(3, 1, 1)
plt.plot(p0)
plt.title("Initial Pressure")
plt.subplot(3, 1, 2)
plt.plot(coeffs_array)
plt.title("Wave Packet Coefficients")
plt.subplot(3, 1, 3)
plt.plot(data_recon - p0)
plt.title("Reconstructed Pressure")
plt.tight_layout()
plt.show()

## Forward Solve

In [ ]:
batch_size = 100
sum_method = "scan_real"
solver = gb_solvers.solve_ODE_base

config = SolverConfig.from_precision()

num_devices = jax.device_count()

mesh = jax.make_mesh((num_devices,), ("x",))

# Create sharding strategy
sharding_strategy = ShardingStrategy(mesh, beam_axis="x")

# Create solver with sharding
msgb_solver = MSGBSolver(
    thr=threshold,
    thr_strat=strategy,
    batch_size=batch_size,
    input_type=input_type,
    ode_solver=solver,
    sum_method=sum_method,
    sharding=sharding_strategy,
)

print(f"\nRunning on {num_devices} devices...")
t1 = time()
gb_multi = msgb_solver.forward(p0, domain, sensors, ts, wpt)[0].block_until_ready()
t2 = time()
print(f"Multi-device: {t2 - t1:.3f}s")

sensor_data = gb_multi.real

N_kw = (N[0], 1)
d = len(N_kw)
dx_kw = (dx[0],) * d
periodic_kw = (periodic[0],) * d

domain_kw = geometry.Domain(N=N_kw, dx=dx_kw, c=c, cfl=cfl, periodic=periodic_kw)

binary_mask_kw = jnp.ones(N_kw)

simulation_options = SimulationOptions(
    data_cast="single",
    smooth_p0=False,
    save_to_disk=True,
)

execution_options = SimulationExecutionOptions(
    is_gpu_simulation=False, delete_data=False, verbose_level=0, show_sim_log=False
)

kwave_solver = KWaveSolver(simulation_options, execution_options)

p0_kw = p0.reshape(N_kw)

t1 = time()
pt_kw = kwave_solver.forward(p0_kw, domain_kw, binary_mask_kw, ts)
t2 = time()
print(f"k-Wave runtime: {t2 - t1:.3f}s")

In [ ]:
def _auto_shared_norm(*arrays, quantile=0.999):
    """Robust shared norm for magnitude panels; guards against rare spikes."""
    vals = np.concatenate([a.ravel() for a in arrays])
    hi = np.quantile(np.abs(vals), quantile)
    vmin, vmax = -hi, hi
    return Normalize(vmin=vmin, vmax=vmax)


def _auto_diff_norm(diff, quantile=0.999):
    """Symmetric diverging norm centered at 0 for difference panel."""
    hi = np.quantile(np.abs(diff), quantile)
    return TwoSlopeNorm(vmin=-hi, vcenter=0.0, vmax=hi)


def _column_index_for_x(profile_pos, extent, Nx):
    x0, x1 = extent[0], extent[1]
    if not (min(x0, x1) <= profile_pos <= max(x0, x1)):
        raise ValueError(f"profile_pos={profile_pos} not within x-range [{x0}, {x1}]")
    # uniform grid assumed
    ix = int(np.clip(np.rint((profile_pos - x0) / ((x1 - x0) / (Nx - 1))), 0, Nx - 1))
    return ix


def plot_msgb_kwave_comparison(
    sensor_data: np.ndarray,
    pt_kw: np.ndarray,
    extent: list[float] | tuple[float, float, float, float],
    ts: np.ndarray,
    profile_pos: float,
    out_png: Path | None = None,
    dpi: int = 300,
    title_prefix: str = "",
    s_norm: Normalize | None = None,
    d_norm: TwoSlopeNorm | None = None,
):
    """
    Build a 2x3 gridspec figure:
      Top: [MSGB | k-Wave | Difference]
      Bottom: time-profile at x = profile_pos
    """
    # --- Validate shapes and semantics ---
    if sensor_data.shape != pt_kw.shape:
        raise ValueError(
            f"Shape mismatch: MSGB {sensor_data.shape} vs k-Wave {pt_kw.shape}"
        )
    Nt, Nx = sensor_data.shape
    if ts.shape[0] != Nt:
        raise ValueError(f"ts length {ts.shape[0]} must equal Nt {Nt}")

    # Compute difference and norms
    diff = pt_kw - sensor_data
    if s_norm is None:
        s_norm = _auto_shared_norm(sensor_data, pt_kw)
    if d_norm is None:
        d_norm = _auto_diff_norm(diff)

    # Profile index
    ix = _column_index_for_x(profile_pos, extent, Nx)

    # --- Figure + layout ---
    fig = plt.figure(figsize=(9, 7))
    gs = fig.add_gridspec(
        nrows=2, ncols=3, height_ratios=[1.0, 0.75], hspace=0.4, wspace=0.15
    )

    ax_msgb = fig.add_subplot(gs[0, 0])
    ax_kw = fig.add_subplot(gs[0, 1])
    ax_dif = fig.add_subplot(gs[0, 2])
    ax_prof = fig.add_subplot(gs[1, :])

    # --- Top row images ---
    # IMPORTANT: we DO NOT transpose; we assume (t, x). If your arrays are (x, t), transpose here consistently.
    im_msgb = ax_msgb.imshow(
        sensor_data, extent=extent, aspect="auto", norm=s_norm, origin="lower"
    )
    im_kw = ax_kw.imshow(  # noqa
        pt_kw, extent=extent, aspect="auto", norm=s_norm, origin="lower"
    )
    im_dif = ax_dif.imshow(
        diff, extent=extent, aspect="auto", norm=d_norm, cmap="RdBu_r", origin="lower"
    )

    # Titles
    ax_msgb.set_title(r"$p_t^{\mathrm{MSGB}}$")
    ax_kw.set_title(r"$p_t^{\mathrm{k\text{-}Wave}}$")
    ax_dif.set_title(r"$p_t^{\mathrm{k\text{-}Wave}} - p_t^{\mathrm{MSGB}}$")

    # Axis labels: only the leftmost gets y-label to reduce clutter
    ax_msgb.set_xlabel(r"$x_s$")
    ax_msgb.set_ylabel("")
    ax_msgb.set_yticks([])
    # ax_msgb.set_ylabel(r"$t$")
    for ax in (ax_kw, ax_dif):
        ax.set_xlabel(r"$x_s$")
        ax.set_yticks([])

    # Remove x ticks from image panels (visual cleaner); the profile axis will carry numeric ticks if desired
    for ax in (ax_msgb, ax_kw, ax_dif):
        ax.set_xticks([])

    # --- Anchored colorbars via axes dividers (robust against resizing) ---
    # Left colorbar for MSGB
    divL = make_axes_locatable(ax_msgb)
    caxL = divL.append_axes("left", size="7%", pad=0.22)
    cbL = fig.colorbar(im_msgb, cax=caxL)
    caxL.yaxis.set_ticks_position("left")
    caxL.yaxis.set_label_position("left")
    cbL.set_label("Amplitude")

    # Right colorbar for Difference
    divR = make_axes_locatable(ax_dif)
    caxR = divR.append_axes("right", size="7%", pad=0.2)
    cbR = fig.colorbar(im_dif, cax=caxR)
    cbR.set_label("Difference")

    # --- Vertical guideline at profile_pos across all top panels ---
    for ax in (ax_msgb, ax_kw, ax_dif):
        ln = ax.axvline(profile_pos, ls="--", lw=1.2, color="k", zorder=5)
        ln.set_path_effects([pe.Stroke(linewidth=2.6, foreground="k"), pe.Normal()])

    # --- Bottom row: time-profile at selected x ---
    y_msgb = sensor_data[:, ix]
    y_kw = pt_kw[:, ix]
    ax_prof.plot(ts, y_kw, label="k-Wave", lw=2)
    ax_prof.plot(ts, y_msgb, "--", label="MSGB", lw=2)
    ax_prof.set_xlabel(r"$t$")
    ax_prof.set_ylabel(r"$p_t(x_s)$")
    ax_prof.set_title(f"Profile at x = {profile_pos:.6g}")
    ax_prof.legend(frameon=False)
    ax_prof.grid(True, alpha=0.4)

    if title_prefix:
        fig.suptitle(title_prefix, y=0.995)

    # Finalize
    if out_png is not None:
        out_png = Path(out_png)
        fig.savefig(out_png, dpi=dpi, bbox_inches="tight")
    plt.show()
    plt.close(fig)


# ---------------- Example usage ----------------
t_min = ts[0]
t_max = ts[-1]
x_min = 0
x_max = domain.grid_size[0]
desired_x = 0.4
plot_msgb_kwave_comparison(
    sensor_data,
    pt_kw,
    extent=[x_min, x_max, t_min, t_max],
    ts=ts,
    profile_pos=desired_x,
    out_png=PLOT_DIR / "report_fig.png",
)